# Imports

In [524]:
from scipy.signal import savgol_filter
import numpy as np

In [236]:
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_colwidth', None)

import warnings
warnings.filterwarnings("ignore")


In [237]:
import ccxt
from ta.trend import MACD
from ta.momentum import RSIIndicator

In [238]:
import plotly.graph_objects as go
from plotly.offline import init_notebook_mode, iplot

init_notebook_mode(connected=True)

# Parameters

In [645]:
# Balance
initial_balance = balance = 100
asset_quantity = 0
stoploss_threshold = 0.1 # 10% loss will lead to a sell order execution 

# Read data

In [646]:
from datetime import timezone, datetime
dt = datetime(2021, 5, 16)
timestamp = dt.replace(tzinfo=timezone.utc).timestamp()
print(timestamp)

1621123200.0


In [647]:
exchange = ccxt.binance()
bars = exchange.fetch_ohlcv('ADA/USDT', timeframe='1m', )#since = (1621123200000)
df = pd.DataFrame(bars[:-1], columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')

In [648]:
closes = [i[4] for i in bars]

In [649]:
df['timestamp'].min()

Timestamp('2021-05-18 12:36:00')

In [650]:
df['timestamp'].max()

Timestamp('2021-05-18 20:54:00')

# MACD

In [479]:
# df['smooth_3'] = savgol_filter(df['close'].tolist(), 101, 2) # window size 51, polynomial order 3
# df['smooth'] = savgol_filter(df['close'].tolist(), 51, 3) # window size 51, polynomial order 3

In [480]:
# macd_df = MACD(close=df['smooth']) # , , window_slow=7, window_fast=5,  window_sign=4

# df['macd'] = macd_df.macd()
# df['signal'] = macd_df.macd_signal()
# df['histogram'] = macd_df.macd_diff()
# df['seclast_histogram'] = df['histogram'].shift()
# df['thirdlast_histogram'] = df['histogram'].shift(2)
# df['fourthlast_histogram'] = df['histogram'].shift(3)
# df['macd_indicator'] = 0
# # df.loc[(df['fourthlast_histogram'] <= 0) & (df['thirdlast_histogram'] >= 0) & (df['seclast_histogram'] >= 0.0005) & (df['histogram'] >= 0.0006) , 'macd_indicator'] = 1 # buy

# df.loc[(df['seclast_histogram'] <= 0) & (df['histogram'] >= 0.00001) , 'macd_indicator'] = 1 # buy

# df.loc[(df['histogram'] <= 0) & (df['seclast_histogram'] >= 0), 'macd_indicator'] = -1 # sell 

# RSI

# Final Verdict

# Order Execution

In [494]:
df['execution'] = 0
df['histogram'] = None
df['smooth'] = None
smooth_history_list = []
second_smoothing_history_list = []
histogram_history_list = []
closes = df['close'].tolist()

In [495]:
window_length = 101
poly_order =  3

In [497]:
in_position = False
round_trips_count = 0
buying_price = 0
cane_count = 0
climb_count = 0
advice = 0
for index in range(len(df)):
#     print('index: ', index)
    try:
        smooth_temp = savgol_filter(closes[:index+1], window_length, poly_order)
    except Exception as e:
        smooth_history_list.append(None)
        smooth_history_list.append(smooth_temp[-1])

    macd_df = MACD(pd.Series(smoothing_history_list))
    hist_list = macd_df.macd_diff().tolist()
    histogram_history_list.append(hist_list[-1])
    df.loc[index, 'histogram'] = hist_list[-1]
    if histogram_history_list[-2] <= 0 and histogram_history_list[-1] >= 0.00001:
        advice = 1
    elif histogram_history_list[-2] >= 0 and histogram_history_list[-1] <= 0:
        advice = -1
    else:
        advice = 0

    current_price = closes[index]

    # Buy order     
    if advice == 1 and not in_position:
        print('buy order executed.')
        df.loc[index, 'execution'] = 1
        asset_quantity = asset_quantity + (balance / current_price)
        balance = 0
        in_position = True
        cane_count = 0
        buying_price = current_price

    # sell order
    elif advice == -1 and in_position:
        print('sell order executed.')
        # sell order
#             if current_price >= buying_price:
        df.loc[index, 'execution'] = -1
        balance = balance + (asset_quantity * current_price)
        print('balance: ', balance)
        asset_quantity = 0 
        in_position = False
        round_trips_count += 1
#             else:
#                 print('current price lower than buying price')
#                 cane_count += 1
#     if (cane_count >= 3):
#         print('Attention required ! Canes detected!')
#         cane_count = 0
#         df.loc[index, 'execution'] = -1
#         balance = balance + (asset_quantity * current_price)
#         asset_quantity = 0 
#         in_position = False
#         round_trips_count += 1
#         break


    # stop loss
    if in_position and (current_price <= buying_price * stoploss_threshold):
        print('stop loss. selling')
        df.loc[index, 'execution'] = -1
        balance = balance + (asset_quantity * current_price)
        asset_quantity = 0 
        in_position = False
        round_trips_count += 1 
                
        
        

index:  100
smooth history list:  [2.18778978 2.19115453 2.19447945 2.1977644  2.20100924 2.20421382
 2.20737801 2.21050167 2.21358464 2.21662679 2.21962797 2.22258804
 2.22550687 2.22838429 2.23122018 2.2340144  2.23676679 2.23947721
 2.24214553 2.2447716  2.24735527 2.24989641 2.25239488 2.25485052
 2.2572632  2.25963277 2.26195909 2.26424203 2.26648143 2.26867715
 2.27082906 2.272937   2.27500084 2.27702043 2.27899564 2.28092631
 2.2828123  2.28465348 2.2864497  2.28820082 2.28990669 2.29156717
 2.29318213 2.29475141 2.29627487 2.29775238 2.29918378 2.30056894
 2.30190772 2.30319996 2.30444554 2.3056443  2.3067961  2.3079008
 2.30895827 2.30996834 2.31093089 2.31184577 2.31271284 2.31353195
 2.31430297 2.31502574 2.31570013 2.31632599 2.31690319 2.31743157
 2.317911   2.31834134 2.31872244 2.31905415 2.31933634 2.31956886
 2.31975158 2.31988434 2.319967   2.31999943 2.31998148 2.31991301
 2.31979387 2.31962392 2.31940303 2.31913104 2.31880781 2.31843321
 2.31800708 2.31752929 2.3169

LinAlgError: SVD did not converge in Linear Least Squares

In [483]:
sub_df = df[df['histogram'].notna()]
initial_price = sub_df.loc[sub_df.index.min(), 'close']
final_price = df[df['execution']==-1].tail(1)['close'].iloc[0]

if balance <= 0:
    print('balance less than zero')
    balance = final_price * asset_quantity

    
algo_percent = 100 * ((balance - initial_balance) / initial_balance) 
print('Final balance is ${} ({}%)'.format(round(balance, 2), round(algo_percent, 2)))

hodl_percent = 100 *((final_price - initial_price) / initial_price)
print('HODL PNL is {}%'.format(round(hodl_percent, 2)))

KeyError: nan

In [472]:
df['histogram'].tolist()

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

# Plot

In [ ]:
buy_df = df[df['execution']==1]
sell_df = df[df['execution']==-1]

In [ ]:
buy_df.shape

In [ ]:
sell_df.shape

In [ ]:
trace0 = go.Scatter(x=df['timestamp'], y=df['close'], line=dict(color="Black"), name='Ticker')
trace1 = go.Scatter(x=buy_df['timestamp'], y=buy_df['close'], line=dict(color="lightgreen"), name='Buy Orders', mode='markers')
trace2 = go.Scatter(x=sell_df['timestamp'], y=sell_df['close'], line=dict(color="darksalmon"), name='Sell Orders', mode='markers')
trace3 = go.Scatter(x=df['timestamp'], y=smooth_history_list, line=dict(color="lightblue"), name='Smooch')


layout0 = go.Layout(
    title="ADA/USDT | Round Trips: {} | Algo: {}% | HODL Gang: {}%".format(round_trips_count, round(algo_percent, 2), round(hodl_percent, 2)),
    xaxis={"title": "Timestamp"}, # "range": [0, 10]
    yaxis={"title": ""},
    template="plotly_white"
)
fig = go.Figure(data=[trace3, trace0, trace1, trace2], layout=layout0)
fig.show()

In [ ]:
trace3 = go.Scatter(x=df['timestamp'], y=df['histogram'], line=dict(color="red"), name='MACD')
# trace4 = go.Scatter(x=df['timestamp'], y=df['signal'], line=dict(color="green"), name='Signal')
trace0 = go.Bar(
    x=df["timestamp"],
    y=df["histogram"],
    name="histo",
#     text=df["txtcol"],
#     textposition="auto",
    marker_color='red',
)

lay0 = go.Layout(
    title="", template="plotly_white"
)

fig0 = go.Figure(data=[trace0], layout=lay0)
iplot(fig0)

In [ ]:
trace3 = go.Scatter(x=df['timestamp'], y=df['macd'], line=dict(color="red"), name='MACD')
trace4 = go.Scatter(x=df['timestamp'], y=df['signal'], line=dict(color="green"), name='Signal')

layout0 = go.Layout(
    title="ADA/USDT",
    xaxis={"title": "Timestamp"}, # "range": [0, 10]
    yaxis={"title": ""},
    template="plotly_white"
)
fig = go.Figure(data=[trace3, trace4], layout=layout0)
fig.show()

# Scratchpad

In [628]:
all_closes = [2.2846, 2.2832, 2.2831, 2.2834, 2.2836, 2.2822, 2.2818, 2.2841, 2.2836, 2.2821, 2.2829, 2.2831, 2.2867, 2.2873, 2.2869, 2.2849, 2.2809, 2.2795, 2.2791, 2.2796, 2.2804, 2.2831, 2.283, 2.2821, 2.2824, 2.282, 2.2831, 2.2829, 2.2829, 2.2826, 2.2822, 2.2825, 2.2833, 2.2817, 2.2806, 2.2814, 2.282, 2.2837, 2.2825, 2.2827, 2.2823, 2.2823, 2.2823, 2.2829, 2.288, 2.2877, 2.2871, 2.2863, 2.2872, 2.2872, 2.2858, 2.2866, 2.2864, 2.2875, 2.2873, 2.2872, 2.2854, 2.2861, 2.2856, 2.2858, 2.2865, 2.2873, 2.2872, 2.2875, 2.2868, 2.2857, 2.2853, 2.2862, 2.2858, 2.2854, 2.282, 2.2823, 2.2807, 2.2799, 2.2809, 2.2795, 2.2815, 2.2805, 2.2801, 2.2807, 2.2805, 2.2811, 2.2812, 2.2812, 2.2801, 2.2802, 2.28, 2.2804, 2.2797, 2.2798, 2.2797, 2.2791, 2.2784, 2.2775, 2.2778, 2.2772, 2.2772, 2.2738, 2.274, 2.2759, 2.276, 2.2769, 2.2768, 2.2765, 2.278, 2.2777, 2.2778, 2.2776, 2.2792, 2.2781, 2.2813, 2.2834, 2.2837, 2.2829, 2.2824, 2.2866, 2.2863, 2.2866, 2.2895, 2.29, 2.2876, 2.2873, 2.2876, 2.2895, 2.2885, 2.2881, 2.2887, 2.2882, 2.2879, 2.289, 2.289, 2.2881, 2.2883, 2.2891, 2.2889, 2.2896, 2.287, 2.2878, 2.287, 2.2877, 2.2877, 2.2878, 2.288, 2.2886, 2.2877, 2.2869, 2.2856, 2.2856, 2.2863, 2.2869, 2.2874, 2.2877, 2.2879, 2.2891, 2.2867, 2.2862, 2.2878, 2.287, 2.2867, 2.2869, 2.2869, 2.2877, 2.2887, 2.288, 2.2885, 2.2885, 2.2882, 2.2882, 2.2868, 2.287, 2.2881, 2.2872, 2.2866, 2.2856, 2.284, 2.2838, 2.2835, 2.283, 2.2837, 2.2836, 2.2833, 2.2829, 2.2825, 2.2829, 2.2823, 2.2838, 2.2828, 2.2821, 2.2825, 2.2823, 2.2824, 2.2822, 2.2829, 2.2838, 2.2836, 2.2833, 2.2828, 2.2816, 2.282, 2.2817, 2.2808, 2.2805, 2.281, 2.2794, 2.279, 2.2781, 2.2792, 2.2791, 2.2791, 2.2776, 2.2782, 2.2779, 2.2758, 2.2759, 2.2766, 2.2759, 2.276, 2.2759, 2.2758, 2.2754, 2.276, 2.2757, 2.2773, 2.2796, 2.2789, 2.2792, 2.2794, 2.2789, 2.2775, 2.2775, 2.2775, 2.2767, 2.2745, 2.2754, 2.2763, 2.2758, 2.275, 2.2754, 2.2738, 2.2748, 2.2733, 2.2729, 2.2731, 2.2731, 2.2728, 2.2729, 2.2733, 2.2728, 2.2723, 2.273, 2.2729, 2.2726, 2.2726, 2.2728, 2.2757, 2.2756, 2.2752, 2.2749, 2.2753, 2.2757, 2.2764, 2.2773, 2.2778, 2.2771, 2.2764, 2.2761, 2.2757, 2.2742, 2.2729, 2.2736, 2.2748, 2.275, 2.2756, 2.2738, 2.2755, 2.277, 2.2782, 2.2777, 2.2778, 2.2778, 2.2776, 2.2771, 2.278, 2.2771, 2.2757, 2.275, 2.2751, 2.279, 2.2816, 2.2802, 2.2777, 2.2796, 2.2815, 2.2815, 2.2814, 2.2804, 2.28, 2.2813, 2.28, 2.2795, 2.2794, 2.2795, 2.279, 2.277, 2.2772, 2.2772, 2.2775, 2.2771, 2.277, 2.2786, 2.2806, 2.2813, 2.2812, 2.2812, 2.2793, 2.2793, 2.2807, 2.2813, 2.2803, 2.2786, 2.2802, 2.2806, 2.2816, 2.2809, 2.2803, 2.2811, 2.281, 2.2804, 2.2803, 2.2751, 2.2751, 2.2746, 2.2734, 2.2746, 2.2748, 2.2734, 2.2733, 2.2723, 2.2723, 2.2747, 2.2747, 2.2757, 2.2753, 2.2743, 2.2747, 2.2738, 2.2726, 2.273, 2.2726, 2.2733, 2.2728, 2.2735, 2.2736, 2.2726, 2.2705, 2.2702, 2.2691, 2.2698, 2.2699, 2.2702, 2.2708, 2.2677, 2.2679, 2.269, 2.2708, 2.2697, 2.2694, 2.2681, 2.268, 2.2699, 2.2696, 2.2688, 2.2705, 2.2691, 2.2675, 2.2673, 2.2649, 2.2633, 2.2626, 2.2626, 2.2623, 2.2613, 2.2592, 2.2598, 2.2612, 2.266, 2.2641, 2.2647, 2.262, 2.2626, 2.2615, 2.2632, 2.2632, 2.2624, 2.2636, 2.2633, 2.2616, 2.2633, 2.2631, 2.262, 2.262, 2.2652, 2.268, 2.2707, 2.2691, 2.266, 2.2658, 2.2641, 2.2634, 2.2662, 2.2686, 2.2677, 2.2612, 2.2623, 2.2605, 2.2649, 2.2619, 2.2639, 2.264, 2.2628, 2.2616, 2.2608, 2.2598, 2.2601, 2.2597, 2.2591, 2.2576, 2.256, 2.2558, 2.253, 2.251, 2.249, 2.2483, 2.248, 2.2501, 2.251, 2.251, 2.252, 2.2509, 2.2548, 2.2532, 2.2533, 2.2513, 2.2501, 2.2512, 2.2503, 2.2505, 2.249, 2.2508, 2.2507, 2.2498, 2.2487, 2.2494, 2.248, 2.2485, 2.2488, 2.2482, 2.2482, 2.2488, 2.2477, 2.2476, 2.2464, 2.2468, 2.247, 2.2488, 2.2473, 2.2468, 2.2469, 2.2458, 2.2466, 2.2462, 2.2471, 2.2453, 2.2458, 2.2462, 2.2459, 2.2456, 2.2453, 2.2439, 2.2449, 2.2453, 2.2462, 2.2455, 2.2451, 2.2461, 2.2455, 2.2461, 2.2464, 2.2453, 2.2478, 2.2474, 2.2478, 2.2478, 2.2488, 2.2486, 2.2492, 2.2498, 2.2493, 2.2491, 2.249, 2.2497, 2.2494, 2.2492, 2.2487, 2.246, 2.2465, 2.2464, 2.2457, 2.2471, 2.248, 2.2474, 2.2478, 2.2478, 2.2485, 2.2473, 2.2466, 2.2464, 2.2466, 2.2466, 2.2489, 2.2475, 2.2487, 2.2484, 2.2471, 2.2485, 2.248, 2.2477, 2.2484, 2.2487, 2.2499, 2.25, 2.2498, 2.2493, 2.2496, 2.2498, 2.2496, 2.2494, 2.2492, 2.2494, 2.2484, 2.2492, 2.2484, 2.2496, 2.25, 2.2499, 2.2508, 2.2498, 2.2507, 2.2508]


In [731]:
all_closes = df['close'].tolist()

In [732]:
from numpy.fft import *
def filter_signal(signal, threshold=1e8):
    fourier = rfft(signal)
    frequencies = rfftfreq(signal.size, d=1e-3/signal.size)
    fourier[frequencies > threshold] = 0
    return irfft(fourier)

In [733]:
in_position = False
savgol_list = []
rolling_avg_list = []
fft_list = []

In [734]:
for i in range(1, len(all_closes) + 1):
    closes = all_closes[:i]
    
    # savgol
    try:
        smooth_closes = savgol_filter(closes, 175, 3)
        savgol_list.append(smooth_closes[-1])
    except Exception as e:
        savgol_list.append(None)
    
    # rolling average
    try:
        numbers_series = pd.Series(closes)
        windows = numbers_series.rolling(50)
        moving_averages = windows.mean()
        moving_averages_list = moving_averages.tolist()
        rolling_avg_list.append(moving_averages_list[-1])
    except Exception as e:
        print('error in rolling average')
        print(e)
        rolling_avg_list.append(None)
        
     
    # FFT
    try:
        if len(closes) >= 120:
            signal = filter_signal(np.array(closes), threshold=1e3)
            fft_list.append(signal[-1])
        else:
            fft_list.append(None)
    except Exception as e:
        print('error in FFT')
        print(e)
        fft_list.append(None)


In [735]:
len(rolling_avg_list)

499

In [736]:
saverage_list = []
for i in range(len(rolling_avg_list)):
    try:
        one_avg = (savgol_list[i] + rolling_avg_list[i]) / 2
        saverage_list.append(one_avg)
    except Exception as e:
#         print(e)
        saverage_list.append(None)

In [737]:
indices = [i for i in range(len(all_closes))]

In [738]:
trace0 = go.Scatter(x=indices, y=all_closes, line=dict(color="Black"), name='Ticker')
trace1 = go.Scatter(x=indices, y=savgol_list, line=dict(color="lightgreen"), name='savgol')
trace2 = go.Scatter(x=indices, y=rolling_avg_list, line=dict(color="darksalmon"), name='rolling avg', )
trace3 = go.Scatter(x=indices, y=fft_list, line=dict(color="yellow"), name='FFT', )
trace4 = go.Scatter(x=indices, y=saverage_list, line=dict(color="red"), name='Saverage')


layout0 = go.Layout(
    title='',
    xaxis={"title": "Timestamp"}, # "range": [0, 10]
    yaxis={"title": ""},
    template="plotly_white"
)
fig = go.Figure(data=[ trace0, trace1, trace2, trace3, trace4], layout=layout0)
fig.show()